<a href="https://colab.research.google.com/github/RazzberryBoy26/SNN-lab/blob/main/SNN_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import tonic
import torch
import tonic.transforms as transforms
from torch.utils.data import DataLoader

# Define the spatial dimensions of the sensor
# N-MNIST was recorded using a physical neuromorphic vision sensor.
# This pulls the hardcoded physical pixel dimensions (usually 34 x 34) and channels (2: ON/OFF).
sample_size = tonic.datasets.NMNIST.sensor_size

# Define the temporal data transformation
# Raw neuromorphic data is just a continuous, asynchronous list of events: (x, y, timestamp, polarity).
# Standard PyTorch Convolutions cannot read lists; they need dense, grid-like tensors.
transform = transforms.Compose([
    # ToFrame slices the continuous stream of events into discrete "frames" based on time.
    # time_window = 3000 means we bin all spikes that occur within a 3000 microsecond (3ms) window
    # into a single frame. This converts our raw events into the discrete Time Steps our LIF neurons need.
    transforms.ToFrame(sensor_size = sample_size, time_window = 3000)
])

# 3. Load the Datasets
# Downloads and applies the 3ms framing transformation to both the train and test splits.
trainset = tonic.datasets.NMNIST(save_to = './data', train = True, transform = transform)
testset = tonic.datasets.NMNIST(save_to = './data', train = False, transform = transform)

# 4. Handle Variable Sequence Lengths
# Unlike standard video, event recordings don't all last the exact same amount of time.
# Some digits might produce 95 time-steps, others 102. PyTorch cannot batch tensors of different sizes.
# PadTensors adds empty "zero" frames to the shorter recordings so every sequence in the batch is equal.
# batch_first=False ensures the tensor shape is (Time, Batch, Channels, Height, Width) instead of (Batch, Time, ...).
# This perfectly aligns with the `for t_step in range(x.size(0)):` time loop in your SpikeNet!
pad_collate = tonic.collation.PadTensors(batch_first = False)

# 5. Create the DataLoaders
# Bundles our framed, padded data into parallel batches of 128 digits to feed into the GPU.
# We shuffle the training data to prevent the network from memorizing sequence order.
train_loader = DataLoader(trainset, batch_size = 128, shuffle = True, collate_fn = pad_collate)
test_loader = DataLoader(testset, batch_size = 128, shuffle = False, collate_fn = pad_collate)

  0%|          | 0/1011893601 [00:00<?, ?it/s]

Extracting ./data/NMNIST/train.zip to ./data/NMNIST


  0%|          | 0/169674850 [00:00<?, ?it/s]

Extracting ./data/NMNIST/test.zip to ./data/NMNIST


In [3]:
# Importing pyTorch and SNNTorch libraries.
import torch.nn as nn
import snntorch as snn
# We define the architecture + the forward pass of our SNN here.
class SpikeNet(nn.Module):
  # The architecture:
  def __init__(self, num_inp, num_out):
    super().__init__()
    self.num_inp = num_inp
    self.num_out = num_out
    # We start with 1 sample being of dimensions (2, 34, 34), and send it to first convolutional block.
    # Convolution upsamples this into expanded no. of channels, specifically 8.
    # Shape becomes (8, 34, 34).
    self.conv1 = nn.Conv2d(in_channels = 2, out_channels = 8, kernel_size = 3, padding = 1, stride = 1)
    # Dropout is standard regularization technique.
    self.drop1 = nn.Dropout2d(0.1)
    # Pooling shrinks the tensor to dimensions (8, 17, 17)
    self.pool1 = nn.MaxPool2d(kernel_size = 2, stride = 2)
    # We apply the LIF neurons to the output signal that results from the first convolution block.
    # We set beta = 0.95, such that the decay of the membrane potential upon receiving an excitatory input is less.
    self.lif1 = snn.Leaky(beta = 0.95)
    # Now, we send the output spike train into our second convolutional block.
    self.conv2 = nn.Conv2d(in_channels = 8, out_channels = 8, kernel_size = 3, padding = 1, stride = 1)
    self.drop2 = nn.Dropout2d(0.1)
    self.pool2 = nn.MaxPool2d(kernel_size = 2, stride = 2)
    # We have the resultant output spike train with each time step of the sample shaped as (8, 8, 8).
    # We send this inside out LIF neuron to get another spike train.
    self.lif2 = snn.Leaky(beta = 0.95)
    # Now we flatten the channels + the spatial plane and send this inside our final linear layer so as to convert this into 10 separate time signals.
    self.fc = nn.Linear(512, num_out)
    # Pass the 10 time signals into LIF neurons to get 10 spike trains.
    # We will apply rate decoding to this afterwards to get rid of the temporal dimension and get 10 logit scores instead.
    self.lif3 = snn.Leaky(beta = 0.95)

  # Now we define the forward pass.
  def forward(self, x):
    # We initialize the potential of the neurons using init_leaky().
    mem1 = self.lif1.init_leaky()
    mem2 = self.lif2.init_leaky()
    mem3 = self.lif3.init_leaky()
    # This list will hold (with temporal time steps) 10 spike trains from 128 samples of a single batch.
    # We will perform rate decoding on this very spike train.
    rec_spike3 = []
    # We want to record the action of our neural network on the input at each and every time step;
    # And append the output of that time step (0 or 1) in the rec_spike3 to form the final output spike train.
    for t_step in range(x.size(0)):
      # We apply all the previous layers as defined in order.
      x_conv1 = self.conv1(x[t_step])
      x_conv1_drop1 = self.drop1(x_conv1)
      x_conv1_drop1_pool1 = self.pool1(x_conv1_drop1)
      spike1, mem1 = self.lif1(x_conv1_drop1_pool1, mem1)
      x_1 = spike1
      x_1_conv2 = self.conv2(x_1)
      x_1_conv2_drop2 = self.drop2(x_1_conv2)
      x_1_conv2_drop2_pool2 = self.pool2(x_1_conv2_drop2)
      spike2, mem2 = self.lif2(x_1_conv2_drop2_pool2, mem2)
      x_12 = spike2
      # We flatten the intermediate output into dimensions compatible with linear layer.
      x_12_flattened = x_12.view(x_12.size(0), -1)
      x_12_fc = self.fc(x_12_flattened)
      # This gives us 1 time step of the spike train.
      spike3, mem3 = self.lif3(x_12_fc, mem3)
      # Append this into our spike train list output and do for each time step.
      rec_spike3.append(spike3)
    # Now, we get the logit scores by using simple summation over the temporal dimension (aka dim = 0);
    # This will fetch us our 10 numbered vector for each of our 128 samples in the batch.
    logit_scores = torch.sum(torch.stack(rec_spike3), dim = 0)
    return logit_scores

In [5]:
# We want GPU to be the functional device.
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# We instantiate the skeleton of the model.
snn_model = SpikeNet(num_inp = 2, num_out = 10)
# We define the loss function we want to use.
loss_fn = nn.CrossEntropyLoss()
# We define the optimizer we will use, and for what set of parameters.
# Standard Adam optim.
optimizer = torch.optim.Adam(snn_model.parameters(), lr = 1e-3, betas = (0.9, 0.999))
# Store the whole framework in the device so used.
snn_model.to(device)

# Now we define the training loops.
# We will have 5 loops / epochs.
num_train_loops = 5
# For ith epoch:
for i in range(num_train_loops):
  # First, we set the model into the training mode.
  # This tells pyTorch that we want the dropout parts to be active.
  snn_model.train()
  # Initialize the metrics
  running_loss = 0.0
  correct_preds = 0
  total_samples = 0
  # Now, we iterate through the train_loader list.
  # The train_loader list will contain all our batches and target points.
  for batch_idx, (inputs, targets) in enumerate(train_loader):
    # Shift them to the functional device.
    inputs = inputs.to(device)
    targets = targets.to(device)
    # We define outputs to be the forward pass of the inputs.
    outputs = snn_model(inputs)
    # Now, calculate loss with the one-hot encoded targets and the outputs.
    loss = loss_fn(outputs, targets)
    # We run the standard backward pass with the optimizer.
    # We do not need to explicitly define surrogate gradients since snnTorch does that for us.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # We append our recorded running loss.
    running_loss += loss.item()
    # Now, we want to record the correct no. of predictions per batch.
    # After each batch, we calculate the no. of correct predictions and print them.
    _, preds = torch.max(outputs, 1)
    correct_preds += (preds == targets).sum().item()
    total_samples += targets.size(0)
    if batch_idx % 100 == 0:
      current_accuracy = (correct_preds / total_samples) * 100
      print(f"Epoch [{i + 1}/{num_train_loops}], Batch [{batch_idx}/{len(train_loader)}], "f"Loss: {loss.item():.4f}, Train Accuracy: {current_accuracy:.2f}%")
  # Similarly we do for each epoch as well.
  epoch_loss = running_loss / len(train_loader)
  epoch_acc = (correct_preds / total_samples) * 100
  print(f"=== Epoch {i + 1} Complete | Average Loss: {epoch_loss:.4f} | Final Accuracy: {epoch_acc:.2f}% ===")

Epoch [1/5], Batch [0/469], Loss: 10.0243, Train Accuracy: 10.16%
Epoch [1/5], Batch [100/469], Loss: 1.0207, Train Accuracy: 43.34%
Epoch [1/5], Batch [200/469], Loss: 0.6467, Train Accuracy: 60.38%
Epoch [1/5], Batch [300/469], Loss: 0.4415, Train Accuracy: 67.74%
Epoch [1/5], Batch [400/469], Loss: 0.3703, Train Accuracy: 72.24%
=== Epoch 1 Complete | Average Loss: 0.8374 | Final Accuracy: 74.27% ===
Epoch [2/5], Batch [0/469], Loss: 0.3957, Train Accuracy: 87.50%
Epoch [2/5], Batch [100/469], Loss: 0.2763, Train Accuracy: 88.98%
Epoch [2/5], Batch [200/469], Loss: 0.2683, Train Accuracy: 89.83%
Epoch [2/5], Batch [300/469], Loss: 0.3244, Train Accuracy: 90.51%
Epoch [2/5], Batch [400/469], Loss: 0.2923, Train Accuracy: 91.01%
=== Epoch 2 Complete | Average Loss: 0.2750 | Final Accuracy: 91.27% ===
Epoch [3/5], Batch [0/469], Loss: 0.3410, Train Accuracy: 90.62%
Epoch [3/5], Batch [100/469], Loss: 0.3120, Train Accuracy: 93.73%
Epoch [3/5], Batch [200/469], Loss: 0.1386, Train Accur

In [6]:
def evaluate_model(model, test_loader, device):
    # 1. Switch the model to evaluation mode
    # This automatically disables all Dropout layers so they don't randomly kill neurons!
    model.eval()
    correct_predictions = 0
    total_samples = 0

    # 2. Deactivate the gradient engine
    # Since we are not training, this stops PyTorch from calculating derivatives,
    # saving a massive amount of GPU memory and speeding up execution.
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(test_loader):

            # Move data to the same hardware device as the model
            inputs = inputs.to(device)
            targets = targets.to(device)

            # --- FORWARD PASS ---
            # Returns your (Batch_Size, 10) tensor of rate-decoded scores
            outputs = model(inputs)

            # --- CALCULATE ACCURACY ---
            # Find the index of the highest score for each sample in the batch
            # outputs shape: (Batch_Size, 10) -> predicted shape: (Batch_Size)
            _, predicted = torch.max(outputs, dim = 1)

            # Count how many samples are in this batch
            total_samples += targets.size(0)

            # Sum up how many predictions perfectly matched the ground-truth targets
            correct_predictions += (predicted == targets).sum().item()

            # Optional: Print progress updates for large test sets
            if batch_idx % 50 == 0:
                print(f"Processing Test Batch [{batch_idx}/{len(test_loader)}]...")

    # 3. Compute overall accuracy percentage
    final_accuracy = (correct_predictions / total_samples) * 100
    print("\n================ TESTING RESULTS ================")
    print(f"Total Images Evaluated: {total_samples}")
    print(f"Correctly Classified : {correct_predictions}")
    print(f"Final Test Accuracy  : {final_accuracy:.2f}%")
    print("=================================================")

    return final_accuracy

# --- Running the Evaluation ---
# Assuming 'model', 'test_loader', and 'device' are already initialized from your training code:
test_acc = evaluate_model(snn_model, test_loader, device)

Processing Test Batch [0/79]...
Processing Test Batch [50/79]...

================ TESTING RESULTS ================
Total Images Evaluated: 10000
Correctly Classified : 9724
Final Test Accuracy  : 97.24%
